In [3]:
import os
import json
import cv2
import yt_dlp
import numpy as np
from typing import Optional
from IPython.display import display, Image, clear_output

# =====================================================================
# PHASE 1: ALGORITHMIC DATA EXTRACTION & CURATION
# =====================================================================

def is_blurry(image, threshold=100.0):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    variance = cv2.Laplacian(gray, cv2.CV_64F).var()
    return variance < threshold

def is_stationary(prev_frame_gray, curr_frame_gray, threshold=15.0):
    if prev_frame_gray is None: return False
    diff = cv2.absdiff(prev_frame_gray, curr_frame_gray)
    return np.mean(diff) < threshold

def is_scene_duplicate(prev_frame, curr_frame, threshold=0.95):
    if prev_frame is None: return False
    hsv_prev = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2HSV)
    hsv_curr = cv2.cvtColor(curr_frame, cv2.COLOR_BGR2HSV)
    
    hist_prev = cv2.calcHist([hsv_prev], [0, 1], None, [50, 60], [0, 180, 0, 256])
    hist_curr = cv2.calcHist([hsv_curr], [0, 1], None, [50, 60], [0, 180, 0, 256])
    
    cv2.normalize(hist_prev, hist_prev, 0, 1, cv2.NORM_MINMAX)
    cv2.normalize(hist_curr, hist_curr, 0, 1, cv2.NORM_MINMAX)
    
    similarity = cv2.compareHist(hist_prev, hist_curr, cv2.HISTCMP_CORREL)
    return similarity > threshold

def process_youtube_ood(url: str, prefix: str, default_scene: str, default_time: str, default_weather: str, 
                        frame_skip_seconds: int = 5, start_sec: Optional[int] = None, end_sec: Optional[int] = None,
                        blur_threshold: float = 150.0, crop_bottom_ratio: float = 0.15):
    
    print(f"\n🌍 Starting automated curation sequence for: {prefix.upper()}")
    out_img_dir = "data/ood_data/images"
    out_lbl_dir = "data/ood_data/labels"
    os.makedirs(out_img_dir, exist_ok=True)
    os.makedirs(out_lbl_dir, exist_ok=True)
    
    temp_video_path = f"temp_{prefix}.mp4"

    print(f"📥 Downloading video from YouTube...")
    ydl_opts = {'format': 'bestvideo[ext=mp4][height<=1080]+bestaudio[ext=m4a]/best[ext=mp4][height<=1080]', 
                'outtmpl': temp_video_path, 'quiet': True, 'no_warnings': True}
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl: ydl.download([url])
    except Exception as e:
        print(f"⚠️ Error downloading {url}: {e}")
        return

    print(f"🎞️ Slicing video and applying quality filters...")
    cap = cv2.VideoCapture(temp_video_path)
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    if fps == 0: fps = 30 
    frame_skip = fps * frame_skip_seconds 
    
    if start_sec is not None: cap.set(cv2.CAP_PROP_POS_MSEC, start_sec * 1000)
    count = start_sec * fps if start_sec else 0
        
    saved, discarded_blur, discarded_stationary, discarded_repetitive = 0, 0, 0, 0
    last_saved_gray, last_saved_bgr = None, None 

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        if end_sec and (cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.0) > end_sec: break
            
        if count % frame_skip == 0:
            crop_h = int(frame.shape[0] * (1.0 - crop_bottom_ratio))
            frame_cropped = frame[:crop_h, :]
            gray_cropped = cv2.cvtColor(frame_cropped, cv2.COLOR_BGR2GRAY)
            
            if is_blurry(frame_cropped, blur_threshold): discarded_blur += 1
            elif is_stationary(last_saved_gray, gray_cropped, 15.0): discarded_stationary += 1
            elif is_scene_duplicate(last_saved_bgr, frame_cropped, 0.96): discarded_repetitive += 1
            else:
                img_name = f"yt_{prefix}_{saved:04d}.jpg"
                json_name = f"yt_{prefix}_{saved:04d}.json"
                
                frame_resized = cv2.resize(frame_cropped, (1280, 720))
                cv2.imwrite(os.path.join(out_img_dir, img_name), frame_resized)
                
                with open(os.path.join(out_lbl_dir, json_name), 'w') as f:
                    json.dump({"name": img_name, "human_verified": False, 
                               "attributes": {"weather": default_weather, "scene": default_scene, "timeofday": default_time}}, f, indent=4)
                    
                saved += 1
                last_saved_gray, last_saved_bgr = gray_cropped, frame_cropped
        count += 1

    cap.release()
    if os.path.exists(temp_video_path): os.remove(temp_video_path)
    print(f"✅ Success! Extracted {saved} frames. Discarded: {discarded_blur} blur, {discarded_stationary} stop, {discarded_repetitive} repetitive.")

def run_extraction_phase():
    videos = [
        {"url": "https://www.youtube.com/watch?v=4FhheAZ4xSk", "prefix": "tokyo_night", "default_scene": "highway", "default_time": "night", "default_weather": "clear", "start_sec": 35, "end_sec": None},
        {"url": "https://www.youtube.com/watch?v=h9I5YMy07f4", "prefix": "oslo_snow", "default_scene": "city street", "default_time": "night", "default_weather": "snowy", "start_sec": 60, "end_sec": None},
        {"url": "https://www.youtube.com/watch?v=3KgR6_0zz7k", "prefix": "bay_area_rain", "default_scene": "highway", "default_time": "daytime", "default_weather": "rainy", "start_sec": 120, "end_sec": None},
        {"url": "https://www.youtube.com/watch?v=ULWLxBrO8hU", "prefix": "manhattan_dusk", "default_scene": "city street", "default_time": "dawn/dusk", "default_weather": "clear", "start_sec": None, "end_sec": None},
        {"url": "https://www.youtube.com/watch?v=BjmE_8tHXG4", "prefix": "us_fog", "default_scene": "highway", "default_time": "daytime", "default_weather": "foggy", "start_sec": None, "end_sec": None}
    ]
    for v in videos: process_youtube_ood(**v)

# =====================================================================
# PHASE 2: INLINE NOTEBOOK ANNOTATION
# =====================================================================

def run_annotation_phase():
    IMG_DIR, LBL_DIR = "data/ood_data/images", "data/ood_data/labels"
    if not os.path.exists(IMG_DIR):
        print("⚠️ No images found. Run Phase 1 first!")
        return

    WEATHER_MAP = {'c': 'clear', 'r': 'rainy', 's': 'snowy', 'f': 'foggy', 'o': 'overcast'}
    TIME_MAP = {'d': 'daytime', 'n': 'night', 'k': 'dawn/dusk'}
    SCENE_MAP = {'h': 'highway', 'c': 'city street', 'r': 'residential', 't': 'tunnel', 'p': 'parking lot'}

    images = sorted([f for f in os.listdir(IMG_DIR) if f.endswith(('.jpg', '.png'))])
    verified_count = 0

    for img_name in images:
        base_name = os.path.splitext(img_name)[0]
        json_path = os.path.join(LBL_DIR, base_name + ".json")
        
        if not os.path.exists(json_path): continue
        with open(json_path, 'r') as f: data = json.load(f)
        if data.get("human_verified", False): continue 

        attrs = data.get("attributes", {})
        def_w, def_t, def_s = attrs.get("weather", "undefined"), attrs.get("timeofday", "undefined"), attrs.get("scene", "undefined")

        img = cv2.imread(os.path.join(IMG_DIR, img_name))
        if img is None: continue
            
        clear_output(wait=True)
        
        # 1. Print Text Data Above Image
        print("=========================================================")
        print(f"🖼️ FILE: {img_name}   |   STATUS: Needs Verification")
        print(f"🎯 LABELS -> Weather: [{def_w}] | Time: [{def_t}] | Scene: [{def_s}]")
        print("=========================================================\n")
        
        # 2. Render Resized Image (768x432 for compact notebook viewing)
        display_img = cv2.resize(img, (768, 432)) 
        _, encoded_img = cv2.imencode('.png', display_img)
        display(Image(data=encoded_img.tobytes()))
        
        # 3. Handle Interactive Prompts
        print("\n🔥 MAIN COMMAND: [ENTER]=Accept All | [e]=Edit | [q]=Quit")
        cmd = input("Command: ").lower().strip()
        
        if cmd == 'q':
            print("🛑 Quitting session.")
            break
            
        elif cmd == '' or cmd == ' ':
            final_w, final_t, final_s = def_w, def_t, def_s
            print(f"✅ Fast-Accepted: {img_name}")
            
        elif cmd == 'e':
            print("\n✏️ EDIT MODE (Leave blank and press ENTER to keep default)")
            w_in = input(f"  🌤️ Weather [{def_w}] (c=clear, r=rainy, s=snowy, f=foggy, o=overcast): ").lower().strip()
            t_in = input(f"  🌙 Time    [{def_t}] (d=daytime, n=night, k=dawn/dusk): ").lower().strip()
            s_in = input(f"  🛣️ Scene   [{def_s}] (h=highway, c=city street, r=residential, t=tunnel, p=parking lot): ").lower().strip()

            final_w = WEATHER_MAP.get(w_in, def_w) if w_in else def_w
            final_t = TIME_MAP.get(t_in, def_t) if t_in else def_t
            final_s = SCENE_MAP.get(s_in, def_s) if s_in else def_s
            
        else:
            print("⚠️ Invalid command. Skipping image to be safe.")
            continue

        # Save updates
        data["attributes"].update({"weather": final_w, "timeofday": final_t, "scene": final_s})
        data["human_verified"] = True 
        with open(json_path, 'w') as f: json.dump(data, f, indent=4)
        verified_count += 1

    clear_output(wait=True)
    print(f"\n🎉 Session complete! Verified {verified_count} images today.")

# =====================================================================
# MAIN PIPELINE EXECUTION
# =====================================================================

clear_output(wait=True)
print("=====================================================")
print("      OOD DATASET CURATION MASTER PIPELINE           ")
print("=====================================================")
print("1: Phase 1 (Download & Extract Videos)")
print("2: Phase 2 (Human-in-the-loop Notebook Annotation)")
print("3: Run Entire Pipeline (Phase 1 -> Phase 2)")

choice = input("\nEnter your choice [1/2/3]: ").strip()

if choice in ['1', '3']:
    run_extraction_phase()
if choice in ['2', '3']:
    run_annotation_phase()

print("\n🏁 Master Pipeline Execution Finished.")


🎉 Session complete! Verified 537 images today.

🏁 Master Pipeline Execution Finished.
